## Problem

**Problem:**​

Sensors provide live data on stock presence, movement, and depletion for smart restocking.​

When scales are too sensitive, they may trigger unnecessarily and “TOO_MANY,” is recorded.
However, when they are too insensitive, they will fail to trigger and “TOO_FEW,” is recorded.​

Establishing which parameters have an influence on the erroneous measurements could aid
in choosing the optimal sensitivity for weights, which in turn would reduce unnecessary data
transmissions and hence, preserve battery life.​

​

**Limitations:​**

Devices and sections data is biased, with some being more represented than others.​

Data set is imbalanced with regards to sensitivities and error types measured.​

Data set lacks operational data like description of surroundings and accuracy of scale.


Load, clean, and split data

In [ ]:


import sys
sys.path.append('../src')

from data_loader import DataLoader
from plot_utils import PlotUtils
from qlearning_agent import QLearningAgent
from reward_calculator import RewardCalculator
import random
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns



In [ ]:

# Load data
file_path = 'data\processed_measurements_without8.csv'
loader = DataLoader(file_path)
data = loader.load_data()



In [ ]:

# Encode categories
data = loader.encode_categories()

# Split into train and validation sets
train_data, val_data = loader.train_val_split()

*Preprocessing steps included:*​

Filtered sensitivity: 2,4,5,6,7​

Filtered relevant_for_weight_analysis != "False"​

Cleaned all NA values​

Data from 2022.04.07-22.06.17​

Merged with metadata and renamed the columns in a simplistic manner​

item_weight and scale_type are added from metadata, 2 categories named as 6kg and 20kg for scale_type​

Divided date and time into two columns, and added a column referring to the week of the year​

Removed “weighttime”, “minwaittime”, “delay”, “relevant_for_weight_analysis”, “is_relevant_section”​

Dataset divided into training (0.8) & validation (0.2)​

​

In [ ]:

print("Train data shape:", train_data.shape)
print("Validation data shape:", val_data.shape)
print("Unique box_type codes mapping:", dict(enumerate(data['box_type'].astype('category').cat.categories)))


## Q learning:

Q Learning is a model-free reinforcement learning (RL) with epsilon greedy
algorithm (exploitation vs exploration) that helps an agent learn an
optimal policy for decision-making in an unknown environment.​


**Goals:**​

Minimize unnecessary weight measurements
(TOO_MANY/Automatic ratio < 0.5 per week)​

Avoid missing weight measurements (TOO_FEW < 2 per week)​

Q-learning Agent Training:

**States:**​

Each unique combination forms a state that
affects optimal sensitivity​

s= (scale_type,weight_bin, box_type)

**Actions: ​**

The agent selects a sensitivity level for the
accelerometer.  

A= {sensitivity [2,8]}

**Reward:**

Calculated weekly​

States that appear less frequently receive a higher
reward through inverse frequency multiplier to
balance the dataset.​
​

**Episode:**​

One episode corresponds to all weeks in the
dataset , there are ~1000 episodes

**Hyperparameters:**​

alpha = 0.2 ​

gamma = 0.85 ​

epsilon = 1.0 ​

epsilon_min = 0.1 ​

epsilon_decay = 0.9985

While reviewing the literature on state-of-the-art methods in IoT optimization, reinforcement learning—specifically Q-learning— was identified as a promising approach. Recent studies have demonstrated its effectiveness in sensor optimization tasks. Q-learning is particularly well-suited for environments that are dynamic and stochastic, where optimal decisions must be learned through experience rather than predefined models, and where feedback is delayed and cumulative—conditions that closely resemble our own use case in supply chain optimization.​

​
Given these similarities, a Q-learning algorithm was implemented to optimize sensitivity settings across different states defined by combinations of scale_type, weight_bin, and box_type. Since our data was observational rather than real-time, we adapted the method accordingly. In our algorithm, reward function is aimed at optimizing the sensitivity by choosing actions from 1 to 8 for different combinations of states (scale_type, weight_bin, box_type) across weeks, based on the correctness of weight classification. It calculates the absolute difference between the proposed action and observed sensitivities from the dataset rows looping through weeks and episodes. Then, the reward is calculated using the closest actual match of row in that week that corresponds to state and chosen action (to simulate what would happen if that action were used). A significant penalty (-200) is applied for (TOO_FEW), as it represents a more critical error. The reward is stored and updated weekly. To address class imbalance in the dataset, we used an inverse frequency multiplier to ensure less frequent states were appropriately weighted.​
​

Each episode of training simulates all weeks in the dataset and includes all possible states. Actions are selected using an epsilon-greedy strategy—balancing exploration and exploitation—controlled by hyperparameters. Following initial feedback, we enhanced the state definition by including box_type to improve model performance. However, since we had already information on ideal weight-bin sizes from the provided data while we did not have information about ideal bin sizes for item weight, we did not include item weight in this model and kept weight-bins.

In [ ]:
# Define states and actions
fixed_states = [tuple(state) for state in train_data[['scale_type','weight_bin','box_type']].drop_duplicates().values.tolist()]
actions = np.linspace(2,8,7)

# Compute inverse frequencies
too_few_counts = train_data[train_data['weightclassification'] == -5].groupby(['scale_type','weight_bin','box_type']).size()
too_many_counts = train_data[train_data['weightclassification'] == -1].groupby(['scale_type','weight_bin','box_type']).size()

inverse_frequency_too_few = too_few_counts.apply(lambda x: 1/x if x>0 else 0).to_dict()
inverse_frequency_too_many = too_many_counts.apply(lambda x: 1/x if x>0 else 0).to_dict()

# Initialize reward calculator
reward_calc = RewardCalculator(train_data, inverse_frequency_too_few, inverse_frequency_too_many)

# Initialize Q-learning agent
agent = QLearningAgent(fixed_states, actions, reward_calc, num_episodes=1000)

# Train agent
weeks = sorted(train_data['week'].unique())
agent.train(weeks, num_episodes=1000)

# Access trained Q_table and episode_rewards
Q_table = agent.Q_table
episode_rewards = agent.episode_rewards

print("Training complete.")
print("Sample Q_table entry:", list(Q_table.items())[0])


 Post-training evaluation and plotting


*Scale Type:* 1:6kg, 0:20kg​
*Box type:* {0: 'LF 221', 1: 'LF 321', 2: 'SK 2311', 3: 'SK 3521'}​
*Meets Criteria:* 'too_few_counts' < 2 & 'ratioauto' < 0.5​

In [ ]:

from plot_utils import PlotUtils

# 1. Post-training Simulation
val_weeks = val_data['week'].unique()
post_training_simulated = []

for week in val_weeks:
    weekly_data = val_data[val_data['week'] == week]
    for state in Q_table:
        scale_type, weight_bin, box_type = state
        best_sensitivity = max(Q_table[state], key=Q_table[state].get)

        possible_rows = weekly_data[
            (weekly_data['scale_type'] == scale_type) &
            (weekly_data['weight_bin'] == weight_bin) &
            (weekly_data['box_type'] == box_type)
        ]
        if not possible_rows.empty:
            possible_rows = possible_rows.copy()
            possible_rows['sensitivity_diff'] = (possible_rows['sensitivity'] - best_sensitivity).abs()
            closest_row = possible_rows.sort_values('sensitivity_diff').iloc[0]
            post_training_simulated.append(closest_row)

post_training_simulated = pd.DataFrame(post_training_simulated)

# 2. Compute meets_criteria for validation & post-training

def compute_meets_criteria(df):
    '''Count too_few and get last ratioauto per week/bin'''
    result = df.groupby(['scale_type','weight_bin','box_type','week']).agg(
        too_few_counts=('weightclassification', lambda x: (x==-5).sum()),
        ratioauto=('ratioauto','last')
    ).reset_index()
    result['meets_criteria'] = (result['too_few_counts'] < 2) & (result['ratioauto'] < 0.5)
    return result

evaluation_df = compute_meets_criteria(val_data)
learning_set1 = compute_meets_criteria(post_training_simulated)

# 3. Episode reward plots
PlotUtils.plot_episode_rewards(agent.episode_rewards, window_size=10)
PlotUtils.plot_q_value_variance(agent.q_value_variance)
PlotUtils.plot_epsilon_decay(agent.epsilon_values)
PlotUtils.plot_weekly_rewards(agent.weekly_rewards)

# 4. Optimal sensitivity table & plots
results = []
for state, actions_dict in Q_table.items():
    best_action = max(actions_dict, key=actions_dict.get)
    results.append([state[0], state[1], state[2], best_action])

results_df = pd.DataFrame(results, columns=['scale_type','weight_bin','box_type','optimal_sensitivity'])

PlotUtils.plot_optimal_sensitivity(results_df, hue_col='scale_type')
PlotUtils.plot_optimal_sensitivity(results_df, hue_col='box_type')
PlotUtils.plot_boxplot_optimal_sensitivity(results_df)

# 5. Validation vs Learning Comparison
PlotUtils.plot_meets_criteria_comparison(evaluation_df, learning_set1)

# 6. Sensitivity distribution histograms
weight_bins_to_plot = ['0 - 803', '803 - 1877', '1877 - 3859']
for w_bin in weight_bins_to_plot:
    sns.histplot(data=data[data['weight_bin']==w_bin], x='sensitivity', hue='scale_type', kde=True)
    plt.title(f"Sensitivity Distribution for '{w_bin}' Bin")
    plt.show()

# 7. Countplots for training & learning evaluations
fig, axes = plt.subplots(2, 1, figsize=(12, 8))
sns.countplot(ax=axes[0], x='week', hue='meets_criteria', data=evaluation_df, palette='Set1')
axes[0].set_title("Validation Evaluation: Pre-Training")
axes[0].set_xlabel("Week")
axes[0].set_ylabel("Number of Entries")
axes[0].legend(title='Meets Criteria', labels=['No','Yes'])
PlotUtils.add_counts_to_bars(axes[0])

sns.countplot(ax=axes[1], x='week', hue='meets_criteria', data=learning_set1, palette='Set1')
axes[1].set_title("Learning Evaluation: Post-Training")
axes[1].set_xlabel("Week")
axes[1].set_ylabel("Number of Entries")
axes[1].legend(title='Meets Criteria', labels=['No','Yes'])
PlotUtils.add_counts_to_bars(axes[1])
plt.tight_layout()
plt.show()



## Evaluation 

Q learning algorithm results are visualised together and separately for box type, scale type and weight bins. Learning rate per episode has an increasing trend and  mean squared error for temporal differences of Q values has a decreasing trend. Moving averages are used in the plots for smoothing. These plots show that the agent has learned to maximize the reward.  ​

​
The results suggest that different scale types behave differently. For example, scale_type = 0 and scale_type = 1 show varying sensitivities for the same weight bins and box types. This suggests that Q-learning agent identified that different hardware or setups need different sensitivity tuning. Also, weight bin affects sensitivity. The weight_bin groups (e.g., 803 - 1877 vs 6673 - 100000) have different optimal sensitivities, indicating that the sensitivity to detect changes should be adjusted based on the current inventory level. For example, when weights are low (small bins), the sensitivity might need to be higher (detect small changes), and when weights are large, sensitivity may be lower (ignore noise). ​
​

We also observe that box types matter.  For example, box_type 3 may require a higher sensitivity in some weight bins, possibly because the weight changes from adding/removing items are subtler or more critical. The values of optimal_sensitivity vary. The sensitivity values range from 2.0 to 8.0, indicating in what amount the system should react to weight changes. Higher values mean it reacts more easily to smaller weight changes, whereas lower values mean it requires more significant weight changes before triggering. For a possible interpretation of the results, we can argue that ,for light bins (like 0-803g or 803-1877g), the scale needs to be very sensitive (high sensitivity number) so it can detect small items being added or removed. For heavy bins (like 6673g+), the scale is less sensitive (lower sensitivity number) because small fluctuations could be just noise or irrelevant. For middle weight ranges, too high and too low sensitivities are not preferred to balance between noise and underreacting to true changes. ​


From these results, we see that finding a balanced split for item weight bins, and adding this to the model would be beneficial to understand the smaller/bigger changes in weights. When compared weeks in pre-training in validation data, after learning we observe that almost every week meets the criteria in the validation set. The validation set for post training is simulated from the validation split dataset, using the learned Q-values for corresponding states and matching rows. 

# Final Recommendation


  To start, one can use the sensitivity values suggested by the Q-learning algorithm. These values are optimized per weight bin and box type (LF 221, LF 321, SK 2311, SK 3521) to achieve more accurate and reliable measurements and minimize the redundant measurements. Since each box type is assigned to a specific scale, the scale differences are also reflected in the results. Therefore, the sensitivity values consider both the box and the scale behavior.​
​

Limitations include the imbalanced  data set and the time-consuming nature of hyperparameter and reward function tuning.​

**​Next Steps:** 

Developing a dynamic / real-time implementation of the Q-Learning algorithm.​

Adding item_weight as bins in state.​

Using more advanced algorithms like Deep Q learning and PPO (Proximal Policy Optimization) for increased performance and stability.​

Obtaining an optimal sensitivity for each weight-bin, box-type and scale-type combination.​

​
​
​